# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook provides a template for loading and exploring a dataset using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a Croissant schema URL.

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import pprint

# Define the dataset URL
url = 'https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json'

# Load the dataset metadata
dataset = mlc.Dataset(url)
metadata = dataset.metadata
print(f"{metadata.name}: {metadata.description}\n")print("Citation:", metadata.citeAs) print("Published:", metadata.datePublished) print("License:", metadata.license)print("Keywords:", metadata.keywords[:5])

## 2. Data Overview
Review available record sets, fields, and their IDs.

In [ ]:
# List all record sets using their `@id` fields

def get_recordset_overview(ds):
    recordsets = ds.metadata.recordSet
    print(f"Number of record sets: {len(recordsets)}\n")
    for recordset in recordsets:
        print(f"- RecordSet @id: {recordset['@id']}")
        print(f"  Name        : {recordset.get('name', '<no name>')}")
        print(f"  Description : {recordset.get('description', '<no description>')}")
        # List fields
        if 'field' in recordset:
            print("  Fields:")
            for field in recordset['field']:
                print(f"    * Field @id: {field['@id']}, name: {field.get('name', '')}, dataType: {field.get('dataType', '')}")
        print("")

get_recordset_overview(dataset)

## 3. Data Extraction
Load data from a specific record set into a DataFrame for analysis. Use the record set and field `@id`s from the overview.

In [ ]:
# List record set IDs
record_sets = [
    rs['@id'] for rs in dataset.metadata.recordSet
]
print("RecordSets @id list:", record_sets)

dataframes = {}
for record_set_id in record_sets:
    print(f"Loading records for RecordSet: {record_set_id}")
    records = list(dataset.records(record_set=record_set_id))
    if records:
        dataframes[record_set_id] = pd.DataFrame(records)
        print("Fields:", dataframes[record_set_id].columns.tolist())
        display(dataframes[record_set_id].head())
    else:
        print("No records in this record set.")

## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps, such as filtering records based on specific criteria, normalizing numeric fields, and categorizing data. This section should include operations like removing outliers, transforming data distributions, or grouping data by key attributes to prepare it for further analysis.

In [ ]:
# ----- CONFIGURATION: Fill with the most relevant RecordSet and numeric field @ids from the above printout ----
# You may need to adjust these IDs after viewing section 2/3 above.

# Example record set and fields: (fill in with real @ids revealed in the printout from above)
# If you re-run the above cell, copy the @id strings for actual available fields
main_rs_id = record_sets[0]  # Use the first record set for demonstration
df = dataframes[main_rs_id]

# Show all numeric fields (float/integer columns):
numeric_cols = df.select_dtypes(include=['float', 'int']).columns.tolist()
print("Numeric fields:", numeric_cols)

# Use the first numeric field for demo if present
if numeric_cols:
    numeric_field = numeric_cols[0]
    threshold = df[numeric_field].quantile(0.75)  # top 25% threshold
    filtered_df = df[df[numeric_field] > threshold].copy()
    print(f"Filtered records with {numeric_field} > {threshold:.3f}:")
    display(filtered_df.head())

    # Normalize the numeric field
    filtered_df[f"{numeric_field}_normalized"] = (filtered_df[numeric_field] - filtered_df[numeric_field].mean()) / filtered_df[numeric_field].std()
    print(f"Normalized {numeric_field} for filtered records:")
    display(filtered_df[[numeric_field, f"{numeric_field}_normalized"]].head())

    # Try to group by a categorical field if any
    non_num_cols = df.select_dtypes(include=['object']).columns.tolist()
    group_field = None
    for c in non_num_cols:
        if df[c].nunique() < 15:
            group_field = c
            break
    if group_field:
        grouped_df = filtered_df.groupby(group_field)[numeric_field].mean().to_frame()
        print(f"Grouped mean of {numeric_field} by {group_field}:")
        display(grouped_df.head())
else:
    print("No numeric fields available for EDA.")

## 5. Visualization
Visualize data distributions or relationships between fields in the dataset.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

if numeric_cols:
    plt.figure(figsize=(8,5))
    sns.histplot(df[numeric_field].dropna(), bins=30, kde=True)
    plt.title(f'Distribution of {numeric_field}')
    plt.xlabel(numeric_field)
    plt.ylabel('Count')
    plt.show()

    # Boxplot by group_field if exists
    if group_field:
        plt.figure(figsize=(10,6))
        sns.boxplot(data=df, x=group_field, y=numeric_field)
        plt.title(f'{numeric_field} by {group_field}')
        plt.xticks(rotation=45)
        plt.show()
else:
    print('No numeric fields to visualize.')

## 6. Conclusion
Summarize key findings and observations from the dataset exploration.

In this notebook, we:

- Loaded a mass spectrometry human mast cell proteomics dataset using its Croissant metadata schema via `mlcroissant`.
- Programmatically listed available record sets and fields by their `@id`.
- Loaded all available records into Pandas dataframes and inspected contents.
- Performed basic filtering, normalization, and grouping operations on accessible numeric fields.
- Visualized numeric field distribution and compared means by selected categorical variables.

### Next steps:
- Explore more specific research questions using field `@id` references in filter/groupby conditions.
- Integrate with other proteomics or annotation datasets for bioinformatics research.
- Extend EDA with more advanced visualizations or modeling as appropriate for the data.